# BEBM AIS log Z and Log-Likelihood

Minimal notebook for estimating the partition function of a trained binary EBM with AIS.

This uses the generic EBM AIS path:

$$E_\beta(v) = (1 - \beta) E_{\mathrm{ref}}(v) + \beta E_{\mathrm{model}}(v),$$

where the reference model is the independent Bernoulli field model.

In [1]:
import torch
import torchvision

from rbms.dataset import load_dataset
from rbms.io import load_model
from rbms.partition_function.ais import compute_partition_function_ais_ebm
from rbms.utils import get_saved_updates

try:
    from torchvision.datasets import MNIST
    from torchvision.transforms import ToTensor
except ImportError:
    MNIST = None
    ToTensor = None

device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.float32

filename = "pcd_trains/BEBM_MNIST_field_h512_ch1024_steps1000_lr1e-3.h5"
train_dataset_name = "data/MNIST_train.h5"
test_dataset_name = "data/MNIST_test.h5"

print(device)


/Users/aidan/Documents/Research_Internship/Code/rbms/rbms/dataset/dataset_class.py:10: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


cpu


In [4]:
train_dataset, test_dataset = load_dataset(
    dataset_name=train_dataset_name,
    test_dataset_name=test_dataset_name,
    device=device,
    dtype=dtype,
)

saved_updates = get_saved_updates(filename)
update = int(saved_updates[-1])

params, chains, _ = load_model(
    filename=filename,
    index=update,
    device=device,
    dtype=dtype,
)

print(f"Loaded update {update}")
print(f"Number of saved updates: {len(saved_updates)}")
print(f"Reference log Z: {params.ref_log_z:.4f}")

Reading dataset from data/MNIST_train.h5...
    Done
Reading dataset from data/MNIST_test.h5...
    Done
Loaded update 50000
Number of saved updates: 1000
Reference log Z: 129.1009


## AIS Estimate

Run several AIS settings and keep the last one as the current `log_z` estimate. If `log_z` keeps increasing strongly as the settings get heavier, the likelihood is still overestimated.


In [3]:
DMALA_ALPHA = 0.5

# Each tuple is: (number of AIS chains, number of inverse-temperature points, MCMC steps per beta).
# Start with the first two rows for a quick check, then add heavier rows when running on GPU.
AIS_SETTINGS = [
    (2_000, 1_000, 1),
    (5_000, 2_000, 1),
    (5_000, 5_000, 1),
    # (5_000, 5_000, 5),
    # (10_000, 10_000, 5),
]

ais_results = []

for seed, (num_chains, num_beta, n_steps) in enumerate(AIS_SETTINGS):
    torch.manual_seed(seed)
    log_z_est = compute_partition_function_ais_ebm(
        num_chains=num_chains,
        num_beta=num_beta,
        params=params,
        n_steps=n_steps,
        kernel="dmala",
        kernel_params={"alpha": DMALA_ALPHA},
    )
    ais_results.append(
        {
            "num_chains": num_chains,
            "num_beta": num_beta,
            "n_steps": n_steps,
            "log_z": log_z_est,
        }
    )
    print(
        f"chains={num_chains:>6}, betas={num_beta:>6}, "
        f"steps={n_steps:>2} -> log Z = {log_z_est:.4f}"
    )

log_z = ais_results[-1]["log_z"]
print(f"\nUsing log Z = {log_z:.4f} from the last AIS setting.")


chains=  2000, betas=  1000, steps= 1 -> log Z = 184.0601
chains=  5000, betas=  2000, steps= 1 -> log Z = 187.8432
chains=  5000, betas=  5000, steps= 1 -> log Z = 196.0020

Using log Z = 196.0020 from the last AIS setting.


## Fixed-Binary Mean Log-Likelihood

For a binary visible sample $v$,

$$\log p(v) = -E(v) - \log Z.$$

This evaluates the fixed binary HDF5 files currently loaded as `train_dataset` and `test_dataset`.


In [4]:
@torch.no_grad()
def weighted_mean_log_likelihood(data, weights, params, log_z, batch_size=2048):
    total = torch.tensor(0.0, device=params.device, dtype=params.dtype)
    normalizer = weights.to(device=params.device, dtype=params.dtype).sum()

    for start in range(0, data.shape[0], batch_size):
        stop = min(start + batch_size, data.shape[0])
        batch = data[start:stop].to(device=params.device, dtype=params.dtype)
        batch_weights = weights[start:stop].to(device=params.device, dtype=params.dtype)

        log_prob = -params.compute_energy_visibles(batch) - log_z
        total += (log_prob * batch_weights).sum()

    return (total / normalizer).item()


ll_train = weighted_mean_log_likelihood(
    data=train_dataset.data,
    weights=train_dataset.weights,
    params=params,
    log_z=log_z,
)

ll_test = weighted_mean_log_likelihood(
    data=test_dataset.data,
    weights=test_dataset.weights,
    params=params,
    log_z=log_z,
)

print(f"Fixed-bin train mean log-likelihood: {ll_train:.4f} nats")
print(f"Fixed-bin test mean log-likelihood:  {ll_test:.4f} nats")
print(f"Fixed-bin train NLL: {-ll_train:.4f} nats")
print(f"Fixed-bin test NLL:  {-ll_test:.4f} nats")


Fixed-bin train mean log-likelihood: -80.7048 nats
Fixed-bin test mean log-likelihood:  -79.7513 nats
Fixed-bin train NLL: 80.7048 nats
Fixed-bin test NLL:  79.7513 nats


## Dynamic-Binarized Mean Log-Likelihood

Dynamic binarization treats each grayscale MNIST image $x \in [0,1]^{784}$ as Bernoulli probabilities and evaluates fresh binary samples

$$v_i \sim \mathrm{Bernoulli}(x_i).$$

The model and `log_z` are unchanged; only the evaluation samples are resampled from grayscale MNIST.


In [2]:
def load_grayscale_mnist(split, root="data/raw_mnist"):
    if MNIST is None or ToTensor is None:
        raise ImportError(
            "torchvision is required for dynamic binarization. "
            "Install it in the notebook environment, then rerun this cell."
        )

    dataset = MNIST(
        root=root,
        train=(split == "train"),
        download=True,
        transform=ToTensor(),
    )

    images = [image.view(-1) for image, _ in dataset]
    return torch.stack(images, dim=0)


gray_train = load_grayscale_mnist("train")
gray_test = load_grayscale_mnist("test")

print(gray_train.shape, gray_train.min().item(), gray_train.max().item())
print(gray_test.shape, gray_test.min().item(), gray_test.max().item())


torch.Size([60000, 784]) 0.0 1.0
torch.Size([10000, 784]) 0.0 1.0


In [7]:
log_z = 196.0020

@torch.no_grad()
def dynamic_binarized_mean_log_likelihood(
    gray_data,
    params,
    log_z,
    num_binarizations=10,
    batch_size=512,
):
    total_log_prob = 0.0
    total_count = 0

    for start in range(0, gray_data.shape[0], batch_size):
        stop = min(start + batch_size, gray_data.shape[0])
        probabilities = gray_data[start:stop].to(
            device=params.device,
            dtype=params.dtype,
        )

        for _ in range(num_binarizations):
            binary = torch.bernoulli(probabilities)
            log_prob = -params.compute_energy_visibles(binary) - log_z
            total_log_prob += log_prob.sum().item()
            total_count += binary.shape[0]

    return total_log_prob / total_count


NUM_DYNAMIC_BINARIZATIONS = 10

torch.manual_seed(0)
ll_train_dyn = dynamic_binarized_mean_log_likelihood(
    gray_data=gray_train,
    params=params,
    log_z=log_z,
    num_binarizations=NUM_DYNAMIC_BINARIZATIONS,
)

torch.manual_seed(1)
ll_test_dyn = dynamic_binarized_mean_log_likelihood(
    gray_data=gray_test,
    params=params,
    log_z=log_z,
    num_binarizations=NUM_DYNAMIC_BINARIZATIONS,
)

print(f"Dynamic-bin train mean log-likelihood: {ll_train_dyn:.4f} nats")
print(f"Dynamic-bin test mean log-likelihood:  {ll_test_dyn:.4f} nats")
print(f"Dynamic-bin train NLL: {-ll_train_dyn:.4f} nats")
print(f"Dynamic-bin test NLL:  {-ll_test_dyn:.4f} nats")


Dynamic-bin train mean log-likelihood: -95.6996 nats
Dynamic-bin test mean log-likelihood:  -94.5472 nats
Dynamic-bin train NLL: 95.6996 nats
Dynamic-bin test NLL:  94.5472 nats


## Optional: AIS Seed Stability

Run this only after choosing one AIS setting. The estimates should be close across seeds if the AIS setting is stable.


In [ ]:
# AIS_NUM_CHAINS, AIS_NUM_BETA, AIS_MCMC_STEPS = AIS_SETTINGS[-1]
# seed_estimates = []
#
# for seed in range(3):
#     torch.manual_seed(seed)
#     estimate = compute_partition_function_ais_ebm(
#         num_chains=AIS_NUM_CHAINS,
#         num_beta=AIS_NUM_BETA,
#         params=params,
#         n_steps=AIS_MCMC_STEPS,
#         kernel="dmala",
#         kernel_params={"alpha": DMALA_ALPHA},
#     )
#     seed_estimates.append(estimate)
#     print(seed, estimate)
#
# print("mean", sum(seed_estimates) / len(seed_estimates))
